# Stage 0 — Initial wallet filtering

Loads raw per-side trades from `data/trades_polygon*/*.parquet` and markets from
`data/markets/*.parquet`, enriches fills with copyable-quantity metrics, selects
the top `TOP_PCT` wallets **globally** by training-period PnL, and exports
grouped trades for that wallet universe as partitioned parquet shards.

Pipeline (enriched shards are the on-disk intermediate between phases):

1. **Token lookup** — mode-filtered token table, cached to parquet.
2. **Enrich** (`enrich_shard`) — per condition: copyable-quantity variants
   (JIT-compiled sliding window) plus `last_condition_trade_ts`; output written
   incrementally to `ENRICHED_TRADES_DIR`.
3. **Select** (`wallet_pnl_shard`) — slim scan of every enriched shard; the
   per-wallet training-PnL dicts are summed across shards in this notebook and
   the **global** top `TOP_PCT` of wallets is selected.
4. **Export** (`group_and_export_shard`) — each enriched shard is re-read and
   filtered to the global wallet set, grouped to one row per
   `tx_hash × wallet × side × token_id` (all periods), and written directly to
   `OUT_DIR`. A selected wallet's trades are exported from *every* shard it
   traded in. Shards are partitioned by the first hex char of `condition_id`
   and a group's token pins it to exactly one shard, so no cross-shard
   re-grouping is needed.

### Data schema notes

Trades are stored as **wallet-perspective rows** (already split per side), with key fields:
`wallet`, `side`, `token_id`, `quantity`, `price`, `usdc_amount`.

So each parquet row is already one wallet-side fill (no additional expansion step needed).

Final P&L per wallet is computed from per-trade P&L:
- `BUY`: `final_value_usdc - trade_value_usdc`
- `SELL`: `trade_value_usdc - final_value_usdc`

where `final_value_usdc = quantity × final_price`, and `final_price` is 1.0 for winning
outcome tokens, 0.0 otherwise.

In [16]:
# TO recalculate from scratch — copy-paste these into a terminal (zsh-safe).
# (kept as comments so running this cell never deletes anything)

# # remove processed markets + caches written by this notebook
# find /Users/vobornij/projects/polymarket/data/markets_processed -maxdepth 1 -name 'token_df_filtered*.parquet' -delete
# # legacy caches from an older pipeline version (no longer read or written)
# rm -f /Users/vobornij/projects/polymarket/data/markets_processed/condition_last_trade*.parquet

# # remove enriched trades (regular + sample dirs)
# find /Users/vobornij/projects/polymarket/data/trades_polygon_enriched* -maxdepth 1 -name 'enriched_*' -delete

# # remove processed trade output (regular + sample dirs)
# find /Users/vobornij/projects/polymarket/data/polygon_trades_processed* -maxdepth 1 -name '*.parquet' -delete

In [17]:
%load_ext autoreload
%autoreload 2

import datetime
import shutil
from concurrent.futures import ProcessPoolExecutor, as_completed
from enum import Enum
from pathlib import Path

import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Parameters

In [ ]:
class MarketFilterMode(Enum):
    DEFAULT = "default"
    SPORTS = "sports"
    CRYPTO = "crypto"

# ── MARKET FILTER MODE SWITCH ────────────────────────────────────────────────
MARKET_FILTER_MODE = MarketFilterMode.SPORTS

# For sports mode, optionally narrow to a specific tag (e.g. "Esports").
# None keeps every market that has tag metadata.
ESPORTS_TAG_FILTER = None

# ── market end_date filter (inclusive) ───────────────────────────────────────
# Only markets whose end_date falls within [START_DATE, END_DATE] are loaded.
START_DATE = datetime.date(2025, 1, 1)
END_DATE   = datetime.date(2027, 4, 5)

# ── train / test split ────────────────────────────────────────────────────────
# Conditions with last trade <= END_TRAIN_TS are training, after are testing.
END_TRAIN_TS = datetime.datetime(2026, 8, 1, tzinfo=datetime.timezone.utc)

# ── copy variants (REQUIRED for enrich_shard) ─────────────────────────────────
# List of (lookback window seconds, fill-price factor). The first entry is the
# primary variant: it drives copyable_pnl and the vol/count export columns.
VARIANTS = [
    (5 * 60, 1.0),    # 5 min  @ price × 1.00  (primary variant)
    # (5 * 60, 0.95),   # 5 min  @ price × 0.95
    # (20 * 60, 1.0),   # 20 min @ price × 1.00
]

# ── wallet selection ─────────────────────────────────────────────────────────
# GLOBAL: rank all wallets by training-period SELECTION_PNL summed across all
# shards and keep the top TOP_PCT fraction. Selected wallets are exported from
# every shard they traded in.
TOP_PCT = 0.015
SELECTION_PNL = "copyable_pnl"   # "copyable_pnl" | "trade_pnl"

# ── smoke-test sampling ──────────────────────────────────────────────────────
# When set, only the first SAMPLE_ROWS rows of each raw/enriched shard are
# processed, and all outputs go to *_sample dirs (real outputs are never
# touched). Set to None for a full run.
SAMPLE_ROWS = None

# ── input / output paths ──────────────────────────────────────────────────────
if MARKET_FILTER_MODE == MarketFilterMode.SPORTS:
    TRADES_DIR  = Path("../../data/trades_polygon_sports")
    ENRICHED_TRADES_DIR = Path("../../data/trades_polygon_enriched_sports")
    OUT_DIR = Path("../../data/polygon_trades_processed_sports")
    TOKEN_DF_CACHE = Path("../../data/markets_processed/token_df_filtered_sports.parquet")
elif MARKET_FILTER_MODE == MarketFilterMode.CRYPTO:
    TRADES_DIR  = Path("../../data/trades_polygon_crypto")
    ENRICHED_TRADES_DIR = Path("../../data/trades_polygon_enriched_crypto")
    OUT_DIR = Path("../../data/polygon_trades_processed_crypto")
    TOKEN_DF_CACHE = Path("../../data/markets_processed/token_df_filtered_crypto.parquet")
else:
    TRADES_DIR  = Path("../../data/trades_polygon")
    ENRICHED_TRADES_DIR = Path("../../data/trades_polygon_enriched")
    OUT_DIR = Path("../../data/polygon_trades_processed")
    TOKEN_DF_CACHE = Path("../../data/markets_processed/token_df_filtered.parquet")

MARKETS_DIR = Path("../../data/markets")

if SAMPLE_ROWS is not None:
    ENRICHED_TRADES_DIR = ENRICHED_TRADES_DIR.with_name(ENRICHED_TRADES_DIR.name + "_sample")
    OUT_DIR = OUT_DIR.with_name(OUT_DIR.name + "_sample")

MAX_WORKERS = 8
print(f"mode={MARKET_FILTER_MODE.value} | global top {TOP_PCT:.2%} by {SELECTION_PNL}"
      f" | variants={VARIANTS} | sample_rows={SAMPLE_ROWS}")
print(f"trades:   {TRADES_DIR}")
print(f"enriched: {ENRICHED_TRADES_DIR}")
print(f"output:   {OUT_DIR}")

mode=sports | global top 1.50% by copyable_pnl | variants=[(300, 1.0)] | sample_rows=None
trades:   ../../data/trades_polygon_sports
enriched: ../../data/trades_polygon_enriched_sports
output:   ../../data/polygon_trades_processed_sports


In [19]:
from polymarket_analysis.data.data_catalogue import load_markets_processed, load_token_lookup_df

token_df = load_token_lookup_df()
print(f"Token lookup rows: {len(token_df):,}")

# ── mode filter (vectorized): allowed condition_id set from market tags ──────
markets_meta = load_markets_processed()[["condition_id", "tags"]]
exploded = markets_meta.explode("tags", ignore_index=True)

def _conds_with_tag(tag: str) -> set[str]:
    return set(exploded.loc[exploded["tags"] == tag, "condition_id"])

listed_conds = set(markets_meta.loc[markets_meta["tags"].notna(), "condition_id"])
if MARKET_FILTER_MODE == MarketFilterMode.SPORTS:
    allowed_conds = listed_conds if ESPORTS_TAG_FILTER is None else _conds_with_tag(ESPORTS_TAG_FILTER)
elif MARKET_FILTER_MODE == MarketFilterMode.CRYPTO:
    allowed_conds = _conds_with_tag("Crypto")
else:
    allowed_conds = listed_conds - _conds_with_tag("Sports") - _conds_with_tag("Crypto")

token_df = token_df[
    token_df["condition_id"].isin(allowed_conds) & token_df["token_winner"].notna()
]
print(f"Token lookup rows after mode filter: {len(token_df):,}")

# Cache the filtered lookup; enrich workers read its token_id column.
TOKEN_DF_CACHE.parent.mkdir(parents=True, exist_ok=True)
token_df.to_parquet(TOKEN_DF_CACHE, index=False)
print(f"Cached token lookup -> {TOKEN_DF_CACHE}")
del markets_meta, exploded
len(token_df)

Token lookup rows: 9,407,618
Token lookup rows after mode filter: 9,407,618
Cached token lookup -> ../../data/markets_processed/token_df_filtered_sports.parquet


9407618

## 2 · Enrich shards

Per shard (parallel workers): read in batches, inner-join with the token lookup
(drops rows of markets missing from the lookup), dictionary-encode id columns,
compute the copyable-quantity variants per condition with the JIT kernel, and
write the enriched rows incrementally. `last_condition_trade_ts` is derived per
condition from the shard itself (conditions never span shards).

In [20]:
from polymarket_analysis.preprocessing.fill_extender import enrich_shard

trade_files = sorted(TRADES_DIR.glob("*.parquet"))
print(f"{len(trade_files)} raw shards in {TRADES_DIR}")

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            enrich_shard, f, ENRICHED_TRADES_DIR, TOKEN_DF_CACHE,
            variants=VARIANTS, sample_rows=SAMPLE_ROWS,
        ): f
        for f in trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        f = futures[fut]
        try:
            fut.result()
        except Exception as e:
            print(f"Error enriching {f.name}: {e}")
        if i % 4 == 0 or i == len(futures):
            print(f"  {i}/{len(futures)} shards enriched")

16 raw shards in ../../data/trades_polygon_sports
24428137 trades in 0.parquet
22743374 trades in 1.parquet
23711317 trades in 2.parquet
24055198 trades in 3.parquet
21,022,127 trades after merging with token_df for 1.parquet
21,698,518 trades after merging with token_df for 2.parquet
21,440,912 trades after merging with token_df for 3.parquet
21,643,381 trades after merging with token_df for 0.parquet
Enriched 1.parquet with copyable_qty variants (21,022,127 rows)
23287034 trades in 4.parquet
Enriched 2.parquet with copyable_qty variants (21,698,518 rows)
22573965 trades in 5.parquet
Enriched 3.parquet with copyable_qty variants (21,440,912 rows)
Enriched 0.parquet with copyable_qty variants (21,643,381 rows)
23094709 trades in 6.parquet
  4/16 shards enriched
23957058 trades in 7.parquet
21,562,992 trades after merging with token_df for 4.parquet
20,964,478 trades after merging with token_df for 5.parquet
20,952,228 trades after merging with token_df for 6.parquet
21,551,609 trades a

## 3 · Phase 1 — global wallet selection

Slim scan (parallel workers) of every enriched shard: per-wallet **training**
P&L for *all* wallets. The dicts are summed across shards here, then the
**global** top `TOP_PCT` cut is applied — so a wallet qualifies on its
whole-dataset P&L and is later exported from every shard it traded in.

In [21]:
from polymarket_analysis.preprocessing.shard_processor import wallet_pnl_shard

enriched_trade_files = sorted(ENRICHED_TRADES_DIR.glob("*.parquet"))
print(f"{len(enriched_trade_files)} enriched shards in {ENRICHED_TRADES_DIR}")

wallet_pnl: dict[str, float] = {}
scan_stats: list[dict] = []

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            wallet_pnl_shard, f,
            end_train_ts=pd.Timestamp(END_TRAIN_TS),
            selection_pnl=SELECTION_PNL,
            variants=VARIANTS,
            sample_rows=SAMPLE_ROWS,
        ): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        f = futures[fut]
        try:
            shard_pnl, stats = fut.result()
        except Exception as e:
            print(f"Error scanning {f.name}: {e}")
            continue
        scan_stats.append(stats)
        for w, pnl in shard_pnl.items():
            wallet_pnl[w] = wallet_pnl.get(w, 0.0) + pnl
        if i % 4 == 0 or i == len(futures):
            print(f"  {i}/{len(futures)} shards scanned | unique wallets so far: {len(wallet_pnl):,}")

total_train_rows = sum(s["train_rows"] for s in scan_stats)
print(f"\nWallets with training trades: {len(wallet_pnl):,} | train rows scanned: {total_train_rows:,}")

pnl_series = pd.Series(wallet_pnl)
del wallet_pnl
threshold = float(pnl_series.quantile(1.0 - TOP_PCT))
final_wallets = set(pnl_series[pnl_series >= threshold].index)

print(f"Global top {TOP_PCT:.2%} threshold ({SELECTION_PNL}): {threshold:.2f} USDC")
print(f"Selected wallets: {len(final_wallets):,} of {len(pnl_series):,}")
print(f"Training {SELECTION_PNL} quantiles:\n"
      f"{pnl_series.quantile([0.5, 0.75, 0.9, 0.98, 0.99])}")
del pnl_series

16 enriched shards in ../../data/trades_polygon_enriched_sports
Scanning shard enriched_2.parquet...
Scanning shard enriched_0.parquet...
Scanning shard enriched_3.parquet...
Scanning shard enriched_1.parquet...
Scanning shard enriched_4.parquet...
Scanning shard enriched_5.parquet...
Scanning shard enriched_6.parquet...
Scanning shard enriched_7.parquet...
  4/16 shards scanned | unique wallets so far: 679,891
Scanning shard enriched_8.parquet...
Scanning shard enriched_9.parquet...
Scanning shard enriched_a.parquet...
Scanning shard enriched_b.parquet...
  8/16 shards scanned | unique wallets so far: 853,760
Scanning shard enriched_c.parquet...
Scanning shard enriched_d.parquet...
Scanning shard enriched_e.parquet...
Scanning shard enriched_f.parquet...
  12/16 shards scanned | unique wallets so far: 959,728
  16/16 shards scanned | unique wallets so far: 1,036,820

Wallets with training trades: 1,036,820 | train rows scanned: 244,430,486
Global top 1.50% threshold (copyable_pnl): 96

## 4 · Phase 2 — grouping and export (global wallet set)

Each enriched shard is re-read and filtered to `final_wallets`: fills are
grouped to one row per `tx_hash × wallet × side × token_id` over **all
periods** (train and test, across every shard the wallet traded in) and
written straight to `OUT_DIR`. Only small per-wallet / per-market aggregates
come back to this process.

In [22]:
from polymarket_analysis.preprocessing.shard_processor import group_and_export_shard

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

wallet_parts: list[pd.DataFrame] = []
market_parts: list[pd.DataFrame] = []
shard_stats: dict[str, dict] = {}

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            group_and_export_shard,
            f,
            OUT_DIR / f"{f.stem.removeprefix('enriched_')}.parquet",
            end_train_ts=pd.Timestamp(END_TRAIN_TS),
            top_wallets=final_wallets,
            variants=VARIANTS,
            sample_rows=SAMPLE_ROWS,
        ): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        f = futures[fut]
        try:
            wallet_part, market_part, stats = fut.result()
        except Exception as e:
            print(f"Error processing {f.name}: {e}")
            continue
        shard_stats[f.stem.removeprefix("enriched_")] = stats
        if not wallet_part.empty:
            wallet_parts.append(wallet_part)
        if not market_part.empty:
            market_parts.append(market_part)
        if i % 4 == 0 or i == len(futures):
            print(f"  {i}/{len(futures)} shards processed | shards with data: {len(shard_stats)}")

total_grouped = sum(s["grouped_rows"] for s in shard_stats.values())
total_train   = sum(s["train_rows"] for s in shard_stats.values())
total_test    = sum(s["test_rows"] for s in shard_stats.values())
print(f"\nProcessed {len(shard_stats)}/{len(enriched_trade_files)} shards"
      f" | grouped rows: {total_grouped:,} (train {total_train:,} / test {total_test:,})")
print(f"Exported wallets (global top {TOP_PCT:.2%} by {SELECTION_PNL}): {len(final_wallets):,}")
print(f"Output -> {OUT_DIR}")

Processing shard enriched_0.parquet...
Processing shard enriched_2.parquet...
Processing shard enriched_1.parquet...
Processing shard enriched_3.parquet...
Processing shard enriched_4.parquet...
Processing shard enriched_5.parquet...
Processing shard enriched_6.parquet...
  4/16 shards processed | shards with data: 4
Processing shard enriched_7.parquet...
Processing shard enriched_8.parquet...
Processing shard enriched_9.parquet...
Processing shard enriched_a.parquet...
Processing shard enriched_b.parquet...
  8/16 shards processed | shards with data: 8
Processing shard enriched_c.parquet...
Processing shard enriched_d.parquet...
Processing shard enriched_e.parquet...
  12/16 shards processed | shards with data: 12
Processing shard enriched_f.parquet...
  16/16 shards processed | shards with data: 16

Processed 16/16 shards | grouped rows: 25,212,960 (train 18,739,638 / test 6,473,322)
Exported wallets (global top 1.50% by copyable_pnl): 15,553
Output -> ../../data/polygon_trades_proce

## 5 · Wallet summary (training period)

Per-wallet aggregates summed across shards. `num_markets` sums exactly because
every condition lives in exactly one shard.

In [23]:
if not wallet_parts:
    raise ValueError("No wallet rows produced — check enrichment/processing logs above")

wallet_summary = (
    pd.concat(wallet_parts, ignore_index=True)
    .groupby("wallet", as_index=False)
    .agg(
        num_markets     = ("num_markets",     "sum"),
        num_trades      = ("num_trades",      "sum"),
        total_cost_usdc = ("total_cost_usdc", "sum"),
        trade_pnl       = ("trade_pnl",       "sum"),
        copyable_pnl    = ("copyable_pnl",    "sum"),
    )
    .sort_values("trade_pnl", ascending=False)
    .reset_index(drop=True)
)
del wallet_parts

print(f"Unique wallets in training data: {len(wallet_summary):,}")
print(f"wallet_summary['trade_pnl'] quantiles:\n"
      f"{wallet_summary['trade_pnl'].quantile([0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99])}")
wallet_summary.head(10)

Unique wallets in training data: 15,553
wallet_summary['trade_pnl'] quantiles:
0.01   -31848.44
0.10    -1019.08
0.25      915.03
0.50     2126.20
0.75     5706.98
0.90    19158.98
0.99   406582.99
Name: trade_pnl, dtype: float64


,wallet,num_markets,num_trades,total_cost_usdc,trade_pnl,copyable_pnl
0,0x09b428f7c2b469786286214aa5c90dd9015f7320,17,14935,34346010.44,8047084.27,5979666.51
1,0x2c335066fe58fe9237c3d3dc7b275c2a034a0563,4108,414545,346035592.53,7171607.87,386277.54
2,0x65b54274eba5c76dee6f0fab18a590653811e82f,7,3130,11146625.00,5067532.93,1422103.08
3,0xed64a7bf029040aa331abc87902434d815ef217d,1,1197,3791774.94,4748881.55,1594217.25
4,0xd1c537b2a7cba8d365e111bffb9de7b205e2cbd0,27,9505,16319183.03,4544645.01,1483994.72
5,0xdb27bf2ac5d428a9c63dbc914611036855a6c56e,1051,136739,72317530.69,4438939.62,267235.15
6,0xf8831548531d56ad6a4331493243c447a827cd1f,27,7433,20728905.11,3994593.41,747800.20
7,0xbddf61af533ff524d27154e589d2d7a81510c684,1054,161760,72343267.74,3871430.62,1579524.46
8,0x7c1ee865a785de4c00ee90ed86a38489fb8bbab3,195,19636,14215791.99,3479281.13,756155.15
9,0x16b29c50f2439faf627209b2ac0c7bbddaa8a881,5503,94942,69643027.54,3408554.38,418008.28


In [24]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")
wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["wallet_count_complement"] = N - wallet_stats["wallet_count"]
wallet_stats["num_trades"]  = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"] = wallet_stats["num_markets"].round(1)
wallet_stats["copyable_pnl"] = wallet_stats["copyable_pnl"].round(2)
wallet_stats["trade_pnl"]    = wallet_stats["trade_pnl"].round(2)
wallet_stats

,wallet_count,num_trades,num_markets,copyable_pnl,trade_pnl,wallet_count_complement
quantile,,,,,,
0.00,16,1.00,1.00,-39718.51,-560315.57,15537
0.01,156,1.00,1.00,-850.81,-31848.44,15397
0.05,778,4.00,1.00,759.81,-3718.21,14775
0.10,1555,8.00,2.00,990.01,-1019.08,13998
0.25,3888,28.00,7.00,1198.88,915.03,11665
0.50,7776,95.00,19.00,1841.18,2126.20,7777
0.75,11665,504.00,82.00,3891.14,5706.98,3888
0.90,13998,2204.00,292.80,10435.50,19158.98,1555
0.95,14775,4935.20,642.00,23407.91,45954.40,778


## 6 · Market-level volume summary

In [25]:
markets_meta = load_markets_processed()[["condition_id", "question", "end_date_iso"]].rename(
    columns={"end_date_iso": "end_date"}
)

market_summary = (
    pd.concat(market_parts, ignore_index=True)
    .merge(markets_meta, on="condition_id", how="left")
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)
del market_parts

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Unique markets: 641,214


,condition_id,num_wallets,num_trades,volume_usdc,final_value_usdc,question,end_date
0,0x9c183f63913cee589ec7a8d584fb7743a541f724edc0...,550,13692,23741954.92,23599427.29,Will Belgium win on 2026-07-01?,2026-07-01T00:00:00Z
1,0xfef657b2f9ed83dd3db24c61d115203e836f9967289c...,506,19644,21353652.93,25448041.99,Will Paris Saint-Germain FC win on 2026-05-30?,2026-05-30T00:00:00Z
2,0xecba7b1db1c698ab73b3572abfcde327a02b51126761...,912,26936,18275829.49,26633395.15,France vs. Spain: Team to Advance,2026-07-14T00:00:00Z
3,0x4b045e092c3166aa29d4864c11dffe7e316daafdac56...,1171,33095,18275726.52,23379675.79,Spain vs. Argentina: Team to Advance,2026-07-19T00:00:00Z
4,0x204d24f3a0f5dd5fca825292bdeab6a97af3978b2caa...,492,33239,15608428.58,17651999.11,Seahawks vs. Patriots,2026-02-08T00:00:00Z
5,0x20fac1c925b7a2fed6b3b2736f47a800b4d0d4001b9d...,730,22676,13634723.98,16784608.92,Will France win on 2026-07-14?,2026-07-14T00:00:00Z
6,0x94e50623d3cfdc1256e16ff372062d42c282bec8af62...,766,24302,12578050.27,18283164.60,Mexico vs. England: Team to Advance,2026-07-06T00:00:00Z
7,0xa947b7c696453a288460a8e12aae31b854ca02d0b879...,818,20593,12367301.17,15537518.17,Will Morocco win on 2026-07-04?,2026-07-04T00:00:00Z
8,0x2037b257e602784d35e800c8c7880f56331791e03aaa...,397,11457,11439296.79,9627500.63,France vs. Morocco: O/U 2.5,2026-07-09T00:00:00Z
9,0xccc07a8317079457585bc88e387b58d2bb94576ee2c9...,885,32725,11199843.44,13732408.79,Norway vs. England: Team to Advance,2026-07-11T00:00:00Z


## 7 · Exported trades preview

Read back from the written output shards (nothing large is held in memory).

In [26]:
out_files = sorted(OUT_DIR.glob("*.parquet"))
print(f"Output shards: {len(out_files)}")
for name in sorted(shard_stats):
    s = shard_stats[name]
    print(f"  {name}.parquet  ({s['grouped_rows']:,} rows | "
          f"{s['wallets_in_shard']:,} of {len(final_wallets):,} wallets present)")

DISPLAY_COLS = [
    "tx_hash", "wallet", "side", "dt", "condition_id", "outcome", "position",
    "total_quantity", "avg_price", "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl",
    "num_fills", "is_train",
]
if out_files:
    display(pd.read_parquet(out_files[0], columns=DISPLAY_COLS).head(200))
else:
    print("No output shards written.")

Output shards: 16
  0.parquet  (1,593,985 rows | 10,395 of 15,553 wallets present)
  1.parquet  (1,502,925 rows | 10,561 of 15,553 wallets present)
  2.parquet  (1,605,487 rows | 10,341 of 15,553 wallets present)
  3.parquet  (1,579,268 rows | 10,392 of 15,553 wallets present)
  4.parquet  (1,570,336 rows | 10,366 of 15,553 wallets present)
  5.parquet  (1,528,739 rows | 10,223 of 15,553 wallets present)
  6.parquet  (1,542,097 rows | 10,284 of 15,553 wallets present)
  7.parquet  (1,588,896 rows | 10,548 of 15,553 wallets present)
  8.parquet  (1,584,735 rows | 10,321 of 15,553 wallets present)
  9.parquet  (1,681,790 rows | 10,653 of 15,553 wallets present)
  a.parquet  (1,543,403 rows | 10,543 of 15,553 wallets present)
  b.parquet  (1,531,746 rows | 9,812 of 15,553 wallets present)
  c.parquet  (1,614,662 rows | 10,310 of 15,553 wallets present)
  d.parquet  (1,542,815 rows | 10,492 of 15,553 wallets present)
  e.parquet  (1,594,441 rows | 10,384 of 15,553 wallets present)
  f.parq

,tx_hash,wallet,side,dt,condition_id,outcome,position,total_quantity,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,trade_pnl,copyable_pnl,num_fills,is_train
0,0xf950fe06f8409f05d5ad6a5029df3ea2156c20e45737...,0x4a64afa45a44a01890c2161be88d2b44751d4430,BUY,2025-01-01 00:00:17+00:00,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,Lakers,122.04,122.04,0.32,False,0.00,39.05,0.00,-39.05,-0.00,1,True
1,0xd40a0282dcc17d28c73203e3ea538431114a901928aa...,0x4fc19a907012ef58717ee7ceb1f89057f71aec92,BUY,2025-01-01 00:33:09+00:00,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,Cavaliers,229.15,229.15,0.65,True,1.00,148.95,229.15,80.20,0.00,1,True
2,0x3fe549772f4a06e41961dce76fd154beeaa451316be3...,0x4fc19a907012ef58717ee7ceb1f89057f71aec92,BUY,2025-01-01 00:46:17+00:00,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,Cavaliers,999.98,770.83,0.65,True,1.00,501.04,770.83,269.79,4.86,1,True
3,0x3fe549772f4a06e41961dce76fd154beeaa451316be3...,0xbde579ead903b9cf5c3fc4be5a0c515464769c38,BUY,2025-01-01 00:46:17+00:00,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,Cavaliers,500.00,500.00,0.65,True,1.00,325.00,500.00,175.00,4.86,2,True
4,0x0472de80248713a5289e3905a89e3a51172de4cffcf9...,0x4a64afa45a44a01890c2161be88d2b44751d4430,BUY,2025-01-01 01:00:25+00:00,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,Lakers,271.47,149.43,0.35,False,0.00,52.30,0.00,-52.30,-32.90,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,0x70b67d7e6baf336278da1491bd169c2a81c573ad16b1...,0xfbf3512a0643af90f3b601e79d54e301877b7094,SELL,2025-01-01 02:12:03+00:00,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,Cavaliers,24984.03,15.96,0.70,True,1.00,11.17,15.96,-4.79,-4.79,1,True
196,0x14ebfeffd96d1adb9e00af0f838c547dddcf7ba45157...,0x4a64afa45a44a01890c2161be88d2b44751d4430,BUY,2025-01-01 02:12:31+00:00,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,Lakers,2319.61,93.43,0.35,False,0.00,32.70,0.00,-32.70,-32.70,1,True
197,0x9e4620c0b024dfedcbd7944e1e0128655aeba61142b5...,0x4a64afa45a44a01890c2161be88d2b44751d4430,BUY,2025-01-01 02:12:55+00:00,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,Lakers,2326.23,6.62,0.32,False,0.00,2.12,0.00,-2.12,-2.12,1,True
198,0xadb84cda2ac2b35be3336953bd79365f70b3b3f1a2e3...,0xcbedcc1f8be4bf7e577170626d73beeaa906aa02,BUY,2025-01-01 02:14:07+00:00,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,Lakers,500.00,500.00,0.36,False,0.00,180.00,0.00,-180.00,-180.00,1,True
